In [ ]:
import cv2
import os
import numpy as np
import time

# Use the XML file from the current folder
xml_path = os.path.join(os.getcwd(), "haarcascade_frontalface_default.xml")

if not os.path.exists(xml_path):
    raise FileNotFoundError(f"XML file not found at: {xml_path}")

# Load the face detector
face_cascade = cv2.CascadeClassifier(xml_path)

# Ask for the person's name
name = input("Enter your name: ").strip()
if not name:
    name = "person"

# Folder where face images will be saved
dataset_dir = "dataset"
person_dir = os.path.join(dataset_dir, name.lower().replace(" ", "_"))
os.makedirs(person_dir, exist_ok=True)

# Open webcam
cap = None
for camera_id in [0, 1]:
    test_cap = cv2.VideoCapture(camera_id)
    if test_cap.isOpened():
        cap = test_cap
        print(f"Using camera {camera_id}")
        break
    test_cap.release()

if cap is None:
    raise RuntimeError("Could not open webcam")

sample_count = 0
max_samples = 10
print(f"Collecting {max_samples} images for {name}. Keep your face visible. Press q to stop.")

frame_count = 0
last_save_time = time.time()

while sample_count < max_samples:
    success, frame = cap.read()
    if not success:
        print("Failed to read frame")
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5)

    if len(faces) > 0:
        (x, y, w, h) = faces[0]
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

        now = time.time()
        if now - last_save_time > 0.5:
            face_roi = gray[y:y + h, x:x + w]
            face_roi = cv2.resize(face_roi, (100, 100))
            file_path = os.path.join(person_dir, f"{sample_count}.jpg")
            cv2.imwrite(file_path, face_roi)
            print(f"Saved: {file_path}")
            sample_count += 1
            last_save_time = now
    else:
        print("No face detected. Move closer to the camera.", end="\r")

    cv2.putText(frame, f"Samples: {sample_count}/{max_samples}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
    cv2.imshow("Collect Faces", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

    frame_count += 1

cap.release()
cv2.destroyAllWindows()

# Train the recognizer
images = []
labels = []
label_names = []

for person_folder in sorted(os.listdir(dataset_dir)):
    person_path = os.path.join(dataset_dir, person_folder)
    if not os.path.isdir(person_path):
        continue

    label_names.append(person_folder)
    for file_name in os.listdir(person_path):
        if file_name.endswith((".jpg", ".png", ".jpeg")):
            img_path = os.path.join(person_path, file_name)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is not None:
                images.append(cv2.resize(img, (100, 100)))
                labels.append(len(label_names) - 1)

if len(images) == 0:
    raise ValueError("No training images were found.")

if not hasattr(cv2, "face"):
    raise ImportError("cv2.face is not available. Install opencv-contrib-python")

recognizer = cv2.face.LBPHFaceRecognizer_create()
recognizer.train(images, np.array(labels, dtype=np.int32))
recognizer.write("face_trainer.yml")

with open("labels.txt", "w") as f:
    f.write("\n".join(label_names))

print("Training complete. Run the next cell for face detection.")


Using camera 0
Saved: dataset/piyush/0.jpger to the camera.
Saved: dataset/piyush/1.jpg
Saved: dataset/piyush/2.jpg
Saved: dataset/piyush/3.jpg
Saved: dataset/piyush/4.jpg
Saved: dataset/piyush/5.jpg
Saved: dataset/piyush/6.jpger to the camera.
Saved: dataset/piyush/7.jpger to the camera.
Saved: dataset/piyush/8.jpger to the camera.
Saved: dataset/piyush/9.jpg
Saved: dataset/piyush/10.jpg
Saved: dataset/piyush/11.jpgr to the camera.
Saved: dataset/piyush/12.jpg
Saved: dataset/piyush/13.jpg
Saved: dataset/piyush/14.jpgr to the camera.
Saved: dataset/piyush/15.jpgr to the camera.
Saved: dataset/piyush/16.jpgr to the camera.
Saved: dataset/piyush/17.jpgr to the camera.
Saved: dataset/piyush/18.jpg
Saved: dataset/piyush/19.jpg
Saved: dataset/piyush/20.jpg
Saved: dataset/piyush/21.jpgr to the camera.
Saved: dataset/piyush/22.jpg
Saved: dataset/piyush/23.jpgr to the camera.
Saved: dataset/piyush/24.jpg
Saved: dataset/piyush/25.jpgr to the camera.
Saved: dataset/piyush/26.jpg
Saved: dataset/p

In [1]:
import cv2
import os

# Load the detector and trainer
xml_path = os.path.join(os.getcwd(), "haarcascade_frontalface_default.xml")
face_cascade = cv2.CascadeClassifier(xml_path)
recognizer = cv2.face.LBPHFaceRecognizer_create()
recognizer.read("face_trainer.yml")

with open("labels.txt", "r") as f:
    labels = [line.strip() for line in f if line.strip()]

cap = cv2.VideoCapture(0)

while True:
    success, frame = cap.read()
    if not success:
        print("Failed to read frame")
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5)

    for (x, y, w, h) in faces:
        face_roi = gray[y:y + h, x:x + w]
        face_roi = cv2.resize(face_roi, (100, 100))

        label_id, _ = recognizer.predict(face_roi)
        label_text = labels[label_id] if 0 <= label_id < len(labels) else "Unknown"

        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.putText(frame, label_text, (x, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    cv2.imshow("Face Recognition", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
